# 5-Dataset Load/Iter/Render Sanity Check

This notebook checks 5 datasets:
- BEDLAM train (`BedlamDatasetV2`)
- 3DPW train (`ThreedpwSmplDataset`)
- 3DPW test (`ThreedpwSmplFullSeqDataset`)
- EMDB test (`EmdbSmplFullSeqDataset`)
- UNI3C aligned (`Uni3CAlignedDatasetV1`)

For each dataset it verifies:
1. Dataset is instantiable and sample is loadable
2. Dataloader iteration loop runs without errors
3. A sample visualization is produced: crop image + bbox/kpts overlay + camera/world SMPLX render


In [1]:
import os
from pathlib import Path
from collections import defaultdict
import numpy as np
from tqdm import tqdm, trange
from einops import einsum
import cv2
from PIL import Image
import lovely_tensors as lt
lt.monkey_patch()
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from hmr4d.dataset.bedlam.bedlam import BedlamDatasetV2
from hmr4d.dataset.bedlam.utils import mid2vname
from hmr4d.dataset.threedpw.threedpw_motion_train import ThreedpwSmplDataset
from hmr4d.dataset.threedpw.threedpw_motion_test import ThreedpwSmplFullSeqDataset
from hmr4d.dataset.emdb.emdb_motion_test import EmdbSmplFullSeqDataset
from hmr4d.dataset.imgfeat_motion.uni3c_aligned import Uni3CAlignedDatasetV1
from hmr4d.datamodule.mocap_trainX_testY import collate_fn

from hmr4d.utils.geo_transform import apply_T_on_points, compute_T_ayfz2ay
from hmr4d.utils.geo.hmr_cam import create_camera_sensor
from hmr4d.utils.vis.cv2_utils import draw_bbx_xys_on_image_batch, draw_kpts_with_conf_batch
from hmr4d.utils.vis.renderer import Renderer, get_global_cameras_static, get_ground_params_from_points
from hmr4d.utils.body_model import BodyModelSMPLX, BodyModelSMPLH

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

viz_dir = Path('outputs/debug/dataset_5way_check')
viz_dir.mkdir(parents=True, exist_ok=True)
print('viz_dir =', viz_dir)


device = cuda
viz_dir = outputs/debug/dataset_5way_check


/home/guangyu/anaconda3/envs/hmr/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
# SMPLX for rendering
smpl = BodyModelSMPLH(
    model_path="inputs/checkpoints/body_models", model_type="smpl",
    gender="neutral", num_betas=10, create_body_pose=False, 
    create_betas=False, create_global_orient=False, create_transl=False,
).to(device)
smpl.eval()

smplx = BodyModelSMPLX(
    model_path='inputs/checkpoints/body_models',
    model_type='smplx',
    gender='neutral',
    num_pca_comps=12,
    flat_hand_mean=False,
).to(device)
smplx.eval()

smplx2smpl = torch.load("hmr4d/utils/body_model/smplx2smpl_sparse.pt").to(device)
J_regressor = torch.load("hmr4d/utils/body_model/smpl_neutral_J_regressor.pt").to(device)

def to_device(x):
    if isinstance(x, torch.Tensor):
        return x.to(device)
    if isinstance(x, dict):
        return {k: to_device(v) for k, v in x.items()}
    return x

def make_K_crop(K_full, bbx, crop_size):
    # bbx = [cx, cy, size]
    K = K_full.clone().float()
    cx_b, cy_b, s = bbx[0].float(), bbx[1].float(), bbx[2].float()
    hs = torch.clamp(0.5 * s, min=1e-6)
    scale = (crop_size - 1) / (2.0 * hs)

    fx = K[0, 0] * scale
    fy = K[1, 1] * scale
    cx = (K[0, 2] - (cx_b - hs)) * scale
    cy = (K[1, 2] - (cy_b - hs)) * scale

    Kc = torch.zeros((3, 3), dtype=K.dtype, device=K.device)
    Kc[0, 0] = fx
    Kc[1, 1] = fy
    Kc[0, 2] = cx
    Kc[1, 2] = cy
    Kc[2, 2] = 1.0
    return Kc

def map_kp2d_to_crop(kp2d_full, bbx, crop_size):
    # kp2d_full: (J,2 or 3) in full-image coords
    kp = kp2d_full.clone().float()
    x, y = kp[:, 0], kp[:, 1]
    cx, cy, s = bbx[0].float(), bbx[1].float(), bbx[2].float()
    hs = torch.clamp(0.5 * s, min=1e-6)
    x0, y0 = cx - hs, cy - hs
    scale = (crop_size - 1) / (2.0 * hs)
    kp[:, 0] = (x - x0) * scale
    kp[:, 1] = (y - y0) * scale
    return kp

def get_frame_index_for_image(sample, img_slot=0):
    idx = sample.get('image_frame_indices', None)
    if idx is None or len(idx) == 0:
        return 0
    return int(idx[img_slot].item())

def select_smpl_params_at(params_dict, t):
    out = {}
    for k, v in params_dict.items():
        if isinstance(v, torch.Tensor) and v.dim() >= 2:
            out[k] = v[t:t+1]
        else:
            out[k] = v
    return out

def render_smpl_on_crop(sample, img_slot=0):
    if sample.get('image', None) is None:
        return None, None
    if sample['image'].shape[0] == 0:
        return None, None

    frame_t = get_frame_index_for_image(sample, img_slot=img_slot)
    crop = sample['image'][img_slot].cpu().numpy().astype(np.uint8)
    H, W = crop.shape[:2]

    bbx = sample['bbx_xys'][frame_t]
    K_full = sample['K_fullimg'][frame_t]
    K_crop = make_K_crop(K_full.to(device), bbx.to(device), H)

    cam_render = None
    world_render = None
    with torch.no_grad():

        if 'smpl_params' in sample and sample['smpl_params'] is not None:
            sp = select_smpl_params_at(sample['smpl_params'], frame_t)
            sp = {k: v.to(device) for k, v in sp.items()}
            verts_c = smpl(**sp).vertices[0]            
            verts_c = apply_T_on_points(verts_c, sample["T_w2c"][frame_t].to(device))
            renderer = Renderer(W, H, device=str(device), faces=smpl.faces, K=K_crop.detach())
            cam_render = renderer.render_mesh(verts_c, background=crop.copy())
            
        if 'smpl_params_c' in sample and sample['smpl_params_c'] is not None:
            sp_c = select_smpl_params_at(sample['smpl_params_c'], frame_t)
            sp_c = {k: v.to(device) for k, v in sp_c.items()}
            verts_c = smplx(**sp_c).vertices[0]
            verts_c  = torch.matmul(smplx2smpl, verts_c)
            
            renderer = Renderer(W, H, device=str(device), faces=smpl.faces, K=K_crop.detach())
            cam_render = renderer.render_mesh(verts_c, background=crop.copy())

        # world render: canonicalized world vertices rendered in a simple preview camera
        if 'smpl_params_w' in sample and sample['smpl_params_w'] is not None and sample['smpl_params_w']['body_pose'].sum().item() != 0:
            sp_w = select_smpl_params_at(sample['smpl_params_w'], frame_t)
            sp_w = {k: v.to(device) for k, v in sp_w.items()}
            verts_w = smplx(**sp_w).vertices
            verts_w = torch.stack([torch.matmul(smplx2smpl, v_) for v_ in verts_w])
            
            offset = einsum(J_regressor, verts_w[0], "j v, v i -> j i")[0]  # (3)
            offset[1] = verts_w[:, :, [1]].min()
            verts_w = verts_w - offset
            T_ay2ayfz = compute_T_ayfz2ay(einsum(J_regressor, verts_w[[0]], "j v, l v i -> l j i"), inverse=True)
            verts_w = apply_T_on_points(verts_w, T_ay2ayfz)
            joints = einsum(J_regressor, verts_w, "j v, l v i -> l j i")
            
            global_R, global_T, global_lights = get_global_cameras_static(
                verts_w.cpu(), beta=2.0, cam_height_degree=20, target_center_height=1.0, vec_rot=0,
            )
            _, _, K = create_camera_sensor(W, H, 24)
            
            renderer_w = Renderer(W, H, device=str(device), faces=smpl.faces, K=K.detach())
            scale, cx, cz = get_ground_params_from_points(joints[:, 0], verts_w)
            renderer_w.set_ground(scale * 1.5, cx, cz)
            color = torch.ones(3).float().cuda() * 0.8
            cameras = renderer_w.create_camera(global_R[0], global_T[0])
            world_render = renderer_w.render_with_ground(verts_w, color[None], cameras, global_lights)

    return cam_render, world_render

def overlay_bbx_kpts_on_crop(sample, img_slot=0):
    if sample.get('image', None) is None:
        return None
    if sample['image'].shape[0] == 0:
        return None

    frame_t = get_frame_index_for_image(sample, img_slot=img_slot)
    crop = sample['image'][img_slot].cpu().numpy().astype(np.uint8)
    H, W = crop.shape[:2]

    # In crop space, bbox is effectively full crop after crop_and_resize
    bbx_crop = torch.tensor([[W / 2.0, H / 2.0, float(min(W, H))]], dtype=torch.float32)
    vis = draw_bbx_xys_on_image_batch(bbx_crop, [crop], thickness=2)[0]

    if 'kp2d' in sample and sample['kp2d'] is not None and sample['kp2d'].shape[0] > frame_t:
        kp_full = sample['kp2d'][frame_t]
        bbx = sample['bbx_xys'][frame_t]
        kp_crop = map_kp2d_to_crop(kp_full, bbx, H)
        if kp_crop.shape[-1] >= 3:
            conf = kp_crop[:, 2:3].clamp(0, 1)
            vis = draw_kpts_with_conf_batch([vis], kp_crop[:, :2][None], conf[None], thickness=2)[0]
        else:
            conf = torch.ones((kp_crop.shape[0], 1), dtype=torch.float32)
            vis = draw_kpts_with_conf_batch([vis], kp_crop[:, :2][None], conf[None], thickness=2)[0]

    return vis


/tmp/ipykernel_1005090/4131586443.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  smplx2smpl = torch.load("hmr4d/utils/body_model/smplx2smpl_sparse.pt").to(device)
/tmp

In [3]:
datasets = {
    'bedlam_train': dict(
        ds=BedlamDatasetV2(load_image=True, load_indices=[0, -1], end_frame_window=10),
        batch_size=2,
        iters=6,
    ),
    'threedpw_train': dict(
        ds=ThreedpwSmplDataset(load_image=True, load_indices=[0, -1]),
        batch_size=2,
        iters=6,
    ),
    'threedpw_test': dict(
        ds=ThreedpwSmplFullSeqDataset(load_image=True, load_indices=[0, -1]),
        batch_size=1,
        iters=4,
    ),
    'emdb_test': dict(
        ds=EmdbSmplFullSeqDataset(split=1, load_image=True, load_indices=[0, -1]),
        batch_size=1,
        iters=4,
    ),
    'uni3c_aligned': dict(
        ds=Uni3CAlignedDatasetV1(load_image=True, load_indices=[0, -1]),
        batch_size=2,
        iters=6,
    ),
}

print('loaded datasets:')
for name, cfg in datasets.items():
    print(f'  {name}: len={len(cfg["ds"])}')


[02/15 16:47:36][INFO] [BEDLAM] Loading from inputs/BEDLAM/hmr4d_support
[02/15 16:47:37][INFO] [BEDLAM] Start loading motion files
[02/15 16:48:26][INFO] [BEDLAM] Motion files loaded. Elapsed: 49.79s
[02/15 16:48:26][INFO] [BEDLAM] 37537 sequences. 
/home/guangyu/patrick/GVHMR/hmr4d/dataset/threedpw/threedpw_motion_train.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We r

loaded datasets:
  bedlam_train: len=37537
  threedpw_train: len=88
  threedpw_test: len=37
  emdb_test: len=17
  uni3c_aligned: len=342


In [ ]:
# 8275, 8355
ds = datasets['bedlam_train']['ds']
idx = 32264
mid = ds.idx2meta[idx]
range1, range2 = ds.mid_to_valid_range[mid]
saved = ds._get_saved_frame_indices(mid)
print(saved)
print(range1, range2)

start_candidates = [f for f in saved if range1 <= f < range1 + ds.end_frame_window]
end_candidates = [f for f in saved if range2 - ds.end_frame_window <= f < range2]
print("start_candidates:", start_candidates)
print("end_candidates:", end_candidates)

In [6]:
window = 10
ds = datasets['bedlam_train']['ds']
loader = DataLoader(
    ds,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    drop_last=False,
    collate_fn=collate_fn,
)

seen = 0
for bi, batch in enumerate(loader):
    if batch['cam_angvel'].shape[1] != 120:
        print(f"Batch {bi} has cam_angvel with shape {batch['cam_angvel'].shape}, expected 120 frames.")
    seen += 1
# counter = 0
# for idx in range(len(ds)):
#     mid = ds.idx2meta[idx]
#     data = ds.motion_files[mid].copy()
#     range1, range2 = ds.mid_to_valid_range[mid]
#     saved = ds._get_saved_frame_indices(mid)
#     saved = sorted({int(f) for f in saved if range1 <= int(f) < range2})
#     # start_missing = [x for x in range(range1, range1 + window) if x not in saved]
#     end_missing = [x for x in range(range2-window, range2) if x not in saved]
#     # if len(start_missing) > 0:
#     #     print(f"Start missing frames for idx={idx}, range1={range1}, range2={range2}, saved_len={len(saved)}, missing={start_missing}")
#     if len(end_missing) > 0:
#         print(f"End missing frames for idx={idx}, range1={range1}, range2={range2}, saved_len={len(saved)}, missing={end_missing}")
#         counter += 1
# print(f"Total {counter} samples with missing end frames within window={window}")

KeyboardInterrupt: 

In [ ]:
# Iteration sanity checks
results = {}

for name, cfg in datasets.items():
    ds = cfg['ds']
    bs = cfg['batch_size']
    iters = cfg['iters']

    print(f'\n[{name}] dataloader iteration check ...')
    loader = DataLoader(
        ds,
        batch_size=bs,
        shuffle=True,
        num_workers=0,
        drop_last=False,
        collate_fn=collate_fn,
    )

    ok = True
    seen = 0
    err_msg = ''

    try:
        for bi, batch in enumerate(loader):
            seen += 1
            if 'image' in batch and batch['image'] is not None:
                img_shape = tuple(batch['image'].shape) if isinstance(batch['image'], torch.Tensor) else 'list/mixed'
            else:
                img_shape = None

            # idx_shape = tuple(batch['image_frame_indices'].shape) if ('image_frame_indices' in batch and isinstance(batch['image_frame_indices'], torch.Tensor)) else None
            print(f'  iter={bi:02d} B={batch.get("B", "?")} image={img_shape} image_frame_indices={batch["image_frame_indices"]}')

            if bi + 1 >= iters:
                break
    except Exception as e:
        ok = False
        err_msg = f'{type(e).__name__}: {e}'

    results[name] = dict(iter_ok=ok, num_iters=seen, error=err_msg)
    print(f'  -> iter_ok={ok}, num_iters={seen}, error={err_msg}')


In [ ]:
from hmr4d.utils.smplx_utils import make_smplx
from hmr4d.utils.geo.hmr_cam import perspective_projection, normalize_kp2d, safely_render_x3d_K, get_bbx_xys

smplx_coco = make_smplx("supermotion_v437coco17")

In [ ]:
cfg = datasets['uni3c_aligned']
ds = cfg['ds']
sample = ds[60]
slot = 1

crop = sample['image'][slot].cpu().numpy().astype(np.uint8)
overlay = overlay_bbx_kpts_on_crop(sample, img_slot=slot)
cam_render, world_render = render_smpl_on_crop(sample, img_slot=slot)

panels = [crop, overlay]
titles = ['image_crop', 'crop+bbx+kpts']

if cam_render is not None:
    panels.append(cam_render)
    titles.append('smplx_camera')
if world_render is not None:
    panels.append(world_render)
    titles.append('smplx_world_preview')

n = len(panels)
fig, axs = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1:
    axs = [axs]
for ax, im, title in zip(axs, panels, titles):
    ax.imshow(im)
    ax.set_title(title)
    ax.axis('off')

fig.suptitle(name)
# out_path = viz_dir / f'{name}_sample_render.png'
fig.tight_layout()
# fig.savefig(out_path, dpi=140)
plt.show()
# plt.close(fig)

In [ ]:
sample

In [ ]:
# Per-dataset sample render
for name, cfg in datasets.items():
    ds = cfg['ds']
    print(f'\n[{name}] sample render ...')

    sample = ds[0]
    if sample.get('image', None) is None or sample['image'].shape[0] == 0:
        print('  image is None or empty, skipping visualization')
        results[name]['render_ok'] = False
        results[name]['render_error'] = 'image is None or empty'
        continue

    try:
        crop = sample['image'][0].cpu().numpy().astype(np.uint8)
        overlay = overlay_bbx_kpts_on_crop(sample, img_slot=0)
        cam_render, world_render = render_smpl_on_crop(sample, img_slot=0)

        panels = [crop, overlay]
        titles = ['image_crop', 'crop+bbx+kpts']

        if cam_render is not None:
            panels.append(cam_render)
            titles.append('smplx_camera')
        if world_render is not None:
            panels.append(world_render)
            titles.append('smplx_world_preview')

        n = len(panels)
        fig, axs = plt.subplots(1, n, figsize=(5*n, 5))
        if n == 1:
            axs = [axs]
        for ax, im, title in zip(axs, panels, titles):
            ax.imshow(im)
            ax.set_title(title)
            ax.axis('off')

        fig.suptitle(name)
        out_path = viz_dir / f'{name}_sample_render.png'
        fig.tight_layout()
        fig.savefig(out_path, dpi=140)
        plt.show()
        plt.close(fig)

        print(f'  saved: {out_path}')
        results[name]['render_ok'] = True
        results[name]['render_error'] = ''
    except Exception as e:
        print(f'  render failed: {type(e).__name__}: {e}')
        results[name]['render_ok'] = False
        results[name]['render_error'] = f'{type(e).__name__}: {e}'


In [ ]:
print('\n===== SUMMARY =====')
all_ok = True
for name, r in results.items():
    iter_ok = r.get('iter_ok', False)
    render_ok = r.get('render_ok', False)
    print(f'{name:15s} iter_ok={iter_ok} render_ok={render_ok}')
    if not iter_ok:
        print('  iter_error :', r.get('error', ''))
    if not render_ok:
        print('  render_err :', r.get('render_error', ''))
    all_ok = all_ok and iter_ok and render_ok

print('all_ok =', all_ok)
if not all_ok:
    raise RuntimeError('Some dataset checks failed; see summary above.')


In [ ]:
# from tqdm import tqdm
# import cv2

# split = 2
# load_indices = [0, -1]
# ds=EmdbSmplFullSeqDataset(split=split, load_image=False, load_indices=load_indices)
# num_items = len(ds)

# def resolve_load_indices(length, indices):
#     out = []
#     for i in indices:
#         j = i if i >= 0 else length + i
#         j = max(0, min(length - 1, j))
#         out.append(int(j))
#     return out

# for idx in tqdm(range(num_items), desc=f"Saving EMDB images (split={split})"):
#     vid, start, end = ds.idx2meta[idx]
#     length = end - start
#     if length <= 0:
#         continue

#     rel = resolve_load_indices(length, load_indices)
#     abs_frames = [start + r for r in rel]
#     print(vid, abs_frames)
    
#     category = vid.split("_")[0]
#     vid_name = "_".join(vid.split("_")[1:])
#     video_path = ds.emdb_root / f"{category}/{vid_name}/{vid}_video.mp4"
#     out_dir = ds.emdb_root / f"{category}/{vid_name}/images"
#     out_dir.mkdir(parents=True, exist_ok=True)

#     cap = cv2.VideoCapture(str(video_path))
#     if not cap.isOpened():
#         print(f"[WARN] cannot open video: {video_path}")
#         continue

#     for fidx in abs_frames:
#         out_path = out_dir / f"{int(fidx):05d}.png"
#         if out_path.exists():
#             continue

#         cap.set(cv2.CAP_PROP_POS_FRAMES, int(fidx))
#         ret, frame = cap.read()
#         if not ret:
#             print(f"[WARN] failed to read frame {fidx} from {video_path}")
#             continue

#         ok = cv2.imwrite(str(out_path), frame)  # BGR->PNG is fine
#         if not ok:
#             print(f"[WARN] failed to save: {out_path}")

#     cap.release()

# print("Done.")